# Resumo de Dados da Camada Bronze

Este notebook automatiza a análise dos arquivos Parquet da camada Bronze, consolidando informações sobre tipos de dados, contagem de registros e estatísticas básicas em um único relatório.

In [ ]:
import os
import pandas as pd
import geopandas as gpd
from pathlib import Path
from tqdm import tqdm

# Configuração de caminhos
BASE_DIR = Path("/home/vinicius/Downloads/estudo/fatec/SABADO-TE-ANALISE-DADOS/notebooks/etl-dados-necessarios")
OUTPUT_DIR = BASE_DIR / "comex/data/resume"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

datasets_paths = {
    "comex_stat": BASE_DIR / "comex/data/bronze/comex_stat",
    "ibama_embargos": BASE_DIR / "ibama/data/bronze/ibama_embargos",
    "pam": BASE_DIR / "pam/data/bronze/pam",
    "pib_vab_agro": BASE_DIR / "pib/data/bronze/pib_vab_agro",
    "ppm_asininos": BASE_DIR / "ppm/data/bronze/ppm_asininos",
    "ppm_bovinos": BASE_DIR / "ppm/data/bronze/ppm_bovinos",
    "ppm_bubalinos": BASE_DIR / "ppm/data/bronze/ppm_bubalinos",
    "ppm_caprinos": BASE_DIR / "ppm/data/bronze/ppm_caprinos",
    "ppm_codornas": BASE_DIR / "ppm/data/bronze/ppm_codornas",
    "ppm_equinos": BASE_DIR / "ppm/data/bronze/ppm_equinos",
    "ppm_galinaceos_total": BASE_DIR / "ppm/data/bronze/ppm_galinaceos_total",
    "ppm_galinhas": BASE_DIR / "ppm/data/bronze/ppm_galinhas",
    "ppm_muar": BASE_DIR / "ppm/data/bronze/ppm_muar",
    "ppm_ovinos": BASE_DIR / "ppm/data/bronze/ppm_ovinos",
    "ppm_suinos_matrizes": BASE_DIR / "ppm/data/bronze/ppm_suinos_matrizes",
    "ppm_suinos_total": BASE_DIR / "ppm/data/bronze/ppm_suinos_total"
}

def analyze_dataset(name, path):
    print(f"Analisando dataset: {name}")
    try:
        # Verifica se o caminho existe
        if not path.exists():
            print(f"  ⚠️ Caminho não encontrado: {path}")
            return None

        # Tenta carregar o dataset usando pandas diretamente no diretório
        # O pandas (com engine 'pyarrow') lida bem com partições e arquivos geoparquet para fins de resumo
        try:
            df = pd.read_parquet(path)
        except Exception:
            # Se falhar (ex: se o diretório estiver vazio ou formato incompatível), tenta pegar o primeiro arquivo
            files = list(path.glob("**/*.parquet")) + list(path.glob("**/*.geoparquet"))
            if not files:
                print(f"  ⚠️ Nenhum arquivo parquet encontrado para {name}")
                return None
            df = pd.read_parquet(files[0])
            
        # Coleta metadados das colunas
        summary = []
        num_rows = len(df)
        
        for col in df.columns:
            # Convertemos para tipos nativos do Python para evitar problemas na exportação do CSV/JSON
            summary.append({
                "dataset": name,
                "total_rows": num_rows,
                "column": col,
                "dtype": str(df[col].dtype),
                "non_null_count": int(df[col].count()),
                "null_count": int(df[col].isna().sum()),
                "unique_values": int(df[col].nunique()),
                "sample_values": str(df[col].dropna().unique()[:5].tolist())
            })
        
        return summary
    except Exception as e:
        print(f"  ❌ Erro ao analisar {name}: {str(e)}")
        return None

all_summaries = []
for name, path in tqdm(datasets_paths.items(), desc="Progresso"):
    ds_summary = analyze_dataset(name, path)
    if ds_summary:
        all_summaries.extend(ds_summary)

# Converte para DataFrame e salva
if all_summaries:
    summary_df = pd.DataFrame(all_summaries)
    output_file = OUTPUT_DIR / "resumo_detalhado_bronze.csv"
    summary_df.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"\n✅ Resumo exportado com sucesso para: {output_file}")
else:
    print("\n❌ Nenhum dado foi processado.")

Progresso:   0%|          | 0/16 [00:00<?, ?it/s]

Analisando dataset: comex_stat


Progresso:   6%|▋         | 1/16 [00:01<00:28,  1.88s/it]

Analisando dataset: ibama_embargos


Progresso:  19%|█▉        | 3/16 [00:03<00:10,  1.19it/s]

Analisando dataset: pam
Analisando dataset: pib_vab_agro
Analisando dataset: ppm_asininos
Analisando dataset: ppm_bovinos


Progresso: 100%|██████████| 16/16 [00:03<00:00,  4.78it/s]

Analisando dataset: ppm_bubalinos
Analisando dataset: ppm_caprinos
Analisando dataset: ppm_codornas
Analisando dataset: ppm_equinos
Analisando dataset: ppm_galinaceos_total
Analisando dataset: ppm_galinhas
Analisando dataset: ppm_muar
Analisando dataset: ppm_ovinos
Analisando dataset: ppm_suinos_matrizes
Analisando dataset: ppm_suinos_total

✅ Resumo exportado com sucesso para: /home/vinicius/Downloads/estudo/fatec/SABADO-TE-ANALISE-DADOS/notebooks/etl-dados-necessarios/comex/data/resume/resumo_detalhado_bronze.csv
